# cutout

> Cutout specific region out of a stacked image

In [ ]:
# | default_exp cutout

In [ ]:
# | exporti

from pathlib import Path

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.visualization.wcsaxes import SphericalCircle


from nicl.utilities import physical_to_angular
from nicl.euclid.data_access import DataAccess
from nicl.euclid.utilities import default_data_path, get_nisp_tile

In [ ]:
# | export

def get_cutout_and_save(image_path, cutout_size, output_path, cutout_name, ext_match=None, ra=None, dec=None):
    """
    Extracts and returns a cutout image from a given larger image.

    Parameters:
    - image_path: Full directory of the input FITS file containing the image data.
    - cutout_size: Side length of the cutout (can be in degrees (u.deg) or other angular units (u.arcmin or u.arcsec), will be converted to degrees).
    - output_path: Directory where the cutout image will be saved.
    - cutout_name: Name of the output file.
    - ext_match : optional string to match in extensions name (e.g. for individual dithers), otherwise will cut out all extensions.
    - ra: (Optional) RA of the cutout center (in degrees). Default is the centre of the image if not provided.
    - dec: (Optional) Dec of the cutout center (in degrees). Default is to read the FITS header value if not provided.

    Returns:
    - position: SkyCoord object representing the center of the cutout.
    - cutout_data: 2D array of the cutout image data.
    - cutout_header: FITS header of the cutout.
    - cutout_wcs: WCS object for the cutout.
    """
        
    with fits.open(image_path, memmap=True) as hdul:
        if len(hudl) == 1:
            ext_list = [0]
        elif ext_match is not None:
            ext_list = [hdu.name for hdu in hdul if ext_match in hdu.name]
        else:
            ext_list = [hdu.name for hdu in hdul]
        output_hdul = fits.HDUList()
        for ext in ext_list:
            hdu = hdul[ext]
            data = hdu.data
            if data:
                header = hdu.header
                wcs = WCS(header)
                if ra is None or dec is None:
                    centre_pixel = (np.array(img.shape) - 1) / 2
                    centre_ra, centre_dec = wcs.pixel_to_world(*centre_pixel)
                    ext_ra = centre_ra if ra is None else ra
                    ext_dec = centre_dec if dec is None else dec
                ext_ra, ext_dec, cutout_size = u.Quantity((ext_ra, ext_dec, cutout_size), u.deg)
                position = SkyCoord(ra=ext_ra, dec=ext_dec, frame='icrs')
                cutout = Cutout2D(data, position, cutout_size, wcs=wcs)
                MatchingHDU = type(hdu)
                out_hdu = MatchingHDU(data=cutout.data, header=cutout.header)
                output_hdul.append(out_hdu)
    fn = f"{cutout_name}.fits"
    fn = Path(output_path, fn)
    output_hdul.writeto(fn, overwrite=True)
    return fn, position

In [ ]:
# | export

def cutout(ra, dec, size, file_type, input_path, output_path, cutout_name, filter, da=None):
    if data_access is None:
        da = DataAccess()
    if file_type == "MER":
        ids = da.find_tiles_for_target(ra, dec, size / 2, fully_contained=True)
    elif file_type == "STK":
        ids = da.find_observations_for_target(ra, dec, size / 2, fully_contained=True)
    else:
        raise ValueError(f"Invalid file_type provided: {file_type}.")
    if len(ids) == 0:
        if file_type == "MER":
            ids = da.find_tiles_for_target(ra, dec, size / 2, fully_contained=False)
        elif file_type == "STK":
            ids = da.find_observations_for_target(ra, dec, size / 2, fully_contained=False)
    if len(ids) == 0:
        raise Error(f"No images found for target.")
    if len(ids) > 1:
        raise NotImplementedError("Automatic stacking of multiple pointings not yet implemented.")
    else:
        try:
            if file_type == "MER":
                fn = get_nisp_tile(tile_index, filter=filter, path=input_path)
            elif file_type == "STK":
                fn = get_nisp_stack(tile_index, filter=filter, path=input_path)
        except FileNotFoundError:
            fn = None
        if fn is None:
            raise NotImplementedError("Automatic stacking of single pointing not yet implemented.")
    cutout_name = f"cutout_name_{filter}"
    fn, position = get_cutout_and_save(fn, size, output_path, cutout_name, ra=ra, dec=dec)
    return fn

In [ ]:
# | export


def show_image(image_data, image_wcs, intervalpc=95, ra=None, dec=None, radius=None, outline_shape=None):
    """
    Displays an image and optionally outlines a region around a given RA, Dec.
    Parameters:
    - image_data: 2D array of the image data to display.
    - image_wcs: WCS object associated with the image.
    - ra: Right Ascension of the center (in degrees) for the region to outline.
    - dec: Declination of the center (in degrees) for the region to outline.
    - radius: radius of the circle which will be outlined (can be 'arcsec', 'arcmin' or 'deg', if none given assumes 'deg'). Alternatively, if 'box' is selected radius defines the half side length.
    - intervalpc: Percentile interval for image normalization (default 95).
    - outline_shape: Shape of the region to outline ('circle' or 'box').

    Warning: If you want to display a random image, without calculating the cutout size as in the framework of this script, 
    ensure the units of the radius are embedded within the given parameter (custom radius *u.deg or *u.arcmin). If not the code assumes it is in degrees. 
    """

    interval = PercentileInterval(intervalpc)
    norm = ImageNormalize(image_data, interval=interval, stretch=AsinhStretch())

    fig, ax = plt.subplots(subplot_kw={'projection': image_wcs})
    im = ax.imshow(image_data, norm=norm, origin='lower', cmap='Greys', interpolation='nearest')

    if ra is not None and dec is not None:
        Centre = SkyCoord(ra=ra * u.degree, dec=dec * u.degree, frame='icrs')

        ax.scatter(Centre.ra.deg, Centre.dec.deg, marker='x', s=50, c='red', transform=ax.get_transform('fk5'))

        if outline_shape == 'circle' and radius is not None:
            circle = SphericalCircle((Centre.ra, Centre.dec), radius, edgecolor='cyan', facecolor='none',
                                     transform=ax.get_transform('fk5'))
            ax.add_patch(circle)
            
        elif outline_shape == 'box' and radius is not None:
            radius_deg = radius.to(u.deg).value if isinstance(radius, u.Quantity) else radius
        
            # Scaling RA offset by cos(Dec) to ensure equal angular side lengths
            ra_offset = radius_deg / np.cos(np.radians(dec))
        
            corners = [
                (ra - ra_offset, dec - radius_deg),  # Bottom-left
                (ra - ra_offset, dec + radius_deg),  # Top-left
                (ra + ra_offset, dec + radius_deg),  # Top-right
                (ra + ra_offset, dec - radius_deg),  # Bottom-right
            ]
        
            poly = Polygon(
                corners, closed=True, edgecolor='yellow', facecolor='none',
                transform=ax.get_transform('fk5')
            )
            ax.add_patch(poly)
            
    plt.colorbar(im, ax=ax, orientation='vertical', label='Pixel Intensity')
    plt.xlabel("RA (J2000)")
    plt.ylabel("Dec (J2000)")

## Example

The following is an example of creating a cutout for a cluster. The cluster info would presumably be read from a table.

In [ ]:
cluster_id = "MCXCJ1806.9+6537"
z = 0.2
ra = 268.8240562 * u.deg
dec = 66.6909137 * u.deg
cutout_radius_physical = 750 * u.kpc
path = default_data_path("Q1_R1_clusters_v0", "cluster_id")
cutout_radius_angular = physical_to_angular(cutout_radius_physical, z)
cutout_name = f"{cluster_id}_MER_cutout"
fn = cutout(ra, dec, cutout_radius_angular, file_type="MER", input_path=path, output_path=path, cutout_name=cutout_name, filter="H")

In [ ]:
image = CCDData.read(fn)
show_image(image.data, image.wcs, centre.ra.value, centre.dec.value)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()